# LINet3 SUN RGB-D Fine-tuning from ScanNet Epoch Checkpoints

**Trains SUN RGB-D 19-category from multiple ScanNet pretrain checkpoints.**

Loops over checkpoints saved at different epochs during ScanNet pretraining
(e.g. epoch 85, 95, 105) to compare transfer learning effectiveness at
different points in the pretraining schedule.

---

## Checklist Before Running:

- [ ] **Enable A100 GPU:** Runtime > Change runtime type > A100
- [ ] **ScanNet epoch checkpoints** exist on Drive (from `colab_LINet3_ScanNet_training.ipynb`)
- [ ] **SUN RGB-D dataset** tar on Drive: `datasets/sunrgbd_19_traintest.tar.gz`
- [ ] **Fill in hyperparameters** in Section 7 (from HPO results)

## 1. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

    # Check if A100
    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\nA100 GPU detected!")
    else:
        print(f"\nWARNING: Expected A100, got {gpu_name}")
else:
    print("\nERROR: No GPU available! Enable GPU in Runtime > Change runtime type")

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

## 3. Clone Repository to Local Disk (Fast I/O)

In [ ]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

os.chdir('/content')

if os.path.exists(LOCAL_REPO_PATH):
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    print(f"Repository updated: {LOCAL_REPO_PATH}")
else:
    !git clone {GITHUB_REPO}
    os.chdir(LOCAL_REPO_PATH)
    print(f"Repository cloned: {LOCAL_REPO_PATH}")

!git log --oneline -5
print("=" * 60)

## 4. Install Dependencies

In [ ]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] kornia thop

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import kornia
import thop

print("All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   kornia: {kornia.__version__}")

## 5. Setup Python Path & Imports

In [ ]:
import sys
import os
import json
import copy
import math
import shutil
import warnings
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from collections import Counter
from sklearn.model_selection import train_test_split

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

# Add project to Python path
project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Model
from src.models.linear_integration.li_net3 import li_resnet18
from src.models.common.model_helpers import load_pretrained_backbone, save_checkpoint

# Datasets
from src.data_utils.sunrgbd_dataset import SUNRGBDDataset

# Training
from src.training.augmentation_config import AugmentationConfig
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler
from src.utils.seed import set_seed

print("All imports successful!")

In [ ]:
SEED = 42
DETERMINISTIC = False
set_seed(SEED, deterministic=DETERMINISTIC)
print(f"Seed: {SEED}, Deterministic: {DETERMINISTIC}")

## 6. Copy SUN RGB-D Dataset to RAM

SUN RGB-D is small (~2-3 GB) -- always loads to `/dev/shm` (RAM disk).

In [ ]:
DRIVE_SUN_TAR = "/content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz"
SUN_DATA_PATH = "/dev/shm/sunrgbd_19_traintest"

print("=" * 60)
print("SUN RGB-D 19-CATEGORY DATASET SETUP")
print("=" * 60)

if Path(SUN_DATA_PATH).exists():
    print(f"Already on local disk: {SUN_DATA_PATH}")
elif Path(DRIVE_SUN_TAR).exists():
    print(f"Extracting from Drive: {DRIVE_SUN_TAR}")
    !tar -xzf {DRIVE_SUN_TAR} -C /dev/shm/
    print(f"Extracted to: {SUN_DATA_PATH}")
else:
    raise FileNotFoundError(f"SUN tar not found: {DRIVE_SUN_TAR}")

# Quick verification
for split in ['train', 'test']:
    split_dir = Path(SUN_DATA_PATH) / split
    if split_dir.exists():
        rgb_count = len(list((split_dir / 'rgb').glob('*.png')))
        print(f"  {split}: {rgb_count} samples")

print("=" * 60)

## 7. Configuration

**IMPORTANT:** Set `SCANNET_CHECKPOINT_DIR` to the epoch_checkpoints directory from your ScanNet training run.

In [ ]:
STREAM_LABELS = {0: 'RGB', 1: 'Depth'}

# ======================== SCANNET CHECKPOINTS ========================
# Point this to the epoch_checkpoints directory from your ScanNet training run
SCANNET_CHECKPOINT_DIR = "/content/drive/MyDrive/linet_checkpoints/scannet_pretrain_XXXXXXXX_XXXXXX/epoch_checkpoints"

# Discover available checkpoints
checkpoint_dir = Path(SCANNET_CHECKPOINT_DIR)
if checkpoint_dir.exists():
    checkpoint_files = sorted(checkpoint_dir.glob("checkpoint_epoch_*.pt"))
    CHECKPOINT_PATHS = {
        int(p.stem.split('_')[-1]): str(p) for p in checkpoint_files
    }
    print(f"Found {len(CHECKPOINT_PATHS)} epoch checkpoints:")
    for epoch, path in sorted(CHECKPOINT_PATHS.items()):
        size_mb = Path(path).stat().st_size / 1024**2
        print(f"  Epoch {epoch}: {path} ({size_mb:.1f} MB)")
else:
    raise FileNotFoundError(
        f"Checkpoint dir not found: {SCANNET_CHECKPOINT_DIR}\n"
        "Run colab_LINet3_ScanNet_training.ipynb first to generate epoch checkpoints."
    )

# ======================== SUN DATASET ========================
SUN_DATASET_CONFIG = {
    'data_root': SUN_DATA_PATH,
    'batch_size': 64,
    'num_workers': 5,
    'num_classes': 19,
    'seed': SEED,
}

SUN_AUGMENTATION_CONFIG = AugmentationConfig(
    rgb_aug_prob=1.0,      # TODO: fill from HPO
    rgb_aug_mag=1.0,       # TODO: fill from HPO
    depth_aug_prob=1.0,    # TODO: fill from HPO
    depth_aug_mag=1.0,     # TODO: fill from HPO
)

# ======================== SUN MODEL ========================
SUN_MODEL_CONFIG = {
    'num_classes': 19,
    'stream_input_channels': [3, 1],  # MUST match ScanNet
    'width_multiplier': 0.75,         # MUST match ScanNet
    'dropout_p': 0.5,     # TODO: fill from HPO
    'device': 'cuda',
    'use_amp': True,
}

# ======================== SUN OPTIMIZER ========================
SUN_OPTIMIZER_CONFIG = {
    'stream_lrs': [7e-05, 1.5e-04],        # TODO: fill from HPO
    'shared_lr': 1.5e-04,                   # TODO: fill from HPO
    'stream_weight_decays': [6e-05, 4e-05], # TODO: fill from HPO
    'integration_weight_decay': 1e-04,      # TODO: fill from HPO
    'stem_lr_multiplier': 1.0,
}

SUN_SCHEDULER_CONFIG = {
    'scheduler_type': 'cosine',
    't_max': 115,                            # TODO: fill from HPO
    's1_eta': 1e-06,                         # TODO: fill from HPO
    's2_eta': 2e-06,                         # TODO: fill from HPO
    'eta_min': 7e-07,                        # TODO: fill from HPO
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2,
}

# ======================== SUN TRAINING ========================
SUN_TRAIN_CONFIG = {
    'epochs': 120,
    'grad_clip_norm': 0.8,             # TODO: fill from HPO
    'early_stopping': False,
    'restore_best_weights': True,
    'stream_monitoring': True,
    'modality_dropout': True,
    'modality_dropout_start': 5,
    'modality_dropout_ramp': 20,
    'modality_dropout_rate': 0.12,     # TODO: fill from HPO
    'label_smoothing': 0.12,           # TODO: fill from HPO
    'gradient_monitoring': True,
    'gradient_log_freq': 0,
    'track_integration_weights': True,
    'integration_snapshot_freq': 10,
    'monitor': 'val_mca',
}

# ======================== TRANSFER CONFIG ========================
TRANSFER_CONFIG = {
    'freeze_backbone_epochs': 0,  # Set > 0 to freeze backbone for N warmup epochs
    'freeze_backbone_lr': 1e-3,
}

print(f"\nSUN Configuration:")
print(f"  Classes: {SUN_MODEL_CONFIG['num_classes']}")
print(f"  Epochs: {SUN_TRAIN_CONFIG['epochs']}")
print(f"  Batch size: {SUN_DATASET_CONFIG['batch_size']}")
print(f"  Checkpoints to evaluate: {sorted(CHECKPOINT_PATHS.keys())}")

## 8. Load SUN Dataset

In [ ]:
print("=" * 60)
print("LOADING SUN RGB-D 19-CATEGORY DATASET (TRAIN/VAL)")
print("=" * 60)

dataset_root = Path(SUN_DATA_PATH)

# Load class names
class_names_file = dataset_root / 'class_names.txt'
if class_names_file.exists():
    with open(class_names_file, 'r') as f:
        sun_class_names = [line.strip() for line in f]
    print(f"Classes ({len(sun_class_names)}): {sun_class_names[:5]}...")

# Create augmented train dataset and non-augmented val dataset (both from train/ split)
train_full_dataset = SUNRGBDDataset(
    data_root=SUN_DATASET_CONFIG['data_root'],
    split='train',
    normalize=True,
    **SUN_AUGMENTATION_CONFIG.to_dict(),
)
val_full_dataset = SUNRGBDDataset(
    data_root=SUN_DATASET_CONFIG['data_root'],
    split='train',
    normalize=True,
)
val_full_dataset.split = 'val'  # Disable augmentation in __getitem__

# 80/20 stratified split
all_labels = train_full_dataset.labels
train_indices, val_indices = train_test_split(
    list(range(len(all_labels))),
    test_size=0.2,
    random_state=SEED,
    stratify=all_labels,
)

train_subset = torch.utils.data.Subset(train_full_dataset, train_indices)
val_subset = torch.utils.data.Subset(val_full_dataset, val_indices)

# Weighted sampler for stratified training
subset_labels = [all_labels[i] for i in train_indices]
label_counts = Counter(subset_labels)
num_train = len(subset_labels)
class_weights = {label: num_train / count for label, count in label_counts.items()}
sample_weights = torch.tensor(
    [class_weights[label] for label in subset_labels], dtype=torch.float32)

g = torch.Generator().manual_seed(SEED)
train_sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights, num_samples=num_train,
    replacement=True, generator=g)

def worker_init_fn(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

sun_train_loader = torch.utils.data.DataLoader(
    train_subset,
    batch_size=SUN_DATASET_CONFIG['batch_size'],
    shuffle=False,
    sampler=train_sampler,
    num_workers=SUN_DATASET_CONFIG['num_workers'],
    prefetch_factor=2,
    persistent_workers=True,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
)

sun_val_loader = torch.utils.data.DataLoader(
    val_subset,
    batch_size=SUN_DATASET_CONFIG['batch_size'],
    shuffle=False,
    num_workers=SUN_DATASET_CONFIG['num_workers'],
    prefetch_factor=2,
    persistent_workers=False,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
)

print(f"\nDataset loaded!")
print(f"  Train: {len(train_subset)} samples ({len(sun_train_loader)} batches)")
print(f"  Val:   {len(val_subset)} samples ({len(sun_val_loader)} batches)")

rgb_batch, depth_batch, label_batch = next(iter(sun_train_loader))
print(f"\nBatch shapes: RGB={rgb_batch.shape}, Depth={depth_batch.shape}, Labels={label_batch.shape}")
print("=" * 60)

---
## 9. Fine-tune SUN for Each Checkpoint

Iterates over each ScanNet epoch checkpoint, creates a fresh model, loads the
pretrained backbone, trains on SUN, and evaluates on the test set. Results are
collected for comparison.

In [ ]:
warnings.filterwarnings(
    'ignore',
    message='The epoch parameter in `scheduler.step\\(\\)` was not necessary',
    category=UserWarning
)

# Master output directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
master_output_dir = f"/content/drive/MyDrive/linet_checkpoints/sun_epoch_ablation_{timestamp}"
Path(master_output_dir).mkdir(parents=True, exist_ok=True)

# Collect results across all checkpoints
all_results = {}

for ckpt_epoch, ckpt_path in sorted(CHECKPOINT_PATHS.items()):
    print("\n" + "#" * 70)
    print(f"# CHECKPOINT EPOCH {ckpt_epoch}")
    print(f"# Source: {ckpt_path}")
    print("#" * 70)

    # Reset seed for reproducibility across runs
    set_seed(SEED, deterministic=DETERMINISTIC)

    # --- Create fresh model ---
    model = li_resnet18(
        num_classes=SUN_MODEL_CONFIG['num_classes'],
        stream_input_channels=SUN_MODEL_CONFIG['stream_input_channels'],
        width_multiplier=SUN_MODEL_CONFIG['width_multiplier'],
        dropout_p=SUN_MODEL_CONFIG['dropout_p'],
        device=SUN_MODEL_CONFIG['device'],
        use_amp=SUN_MODEL_CONFIG['use_amp'],
    )

    # --- Load pretrained backbone ---
    print(f"\nLoading pretrained backbone from epoch {ckpt_epoch}...")
    transfer_info = load_pretrained_backbone(model, ckpt_path)
    print(f"  Loaded: {len(transfer_info['loaded'])} keys, Skipped: {len(transfer_info['skipped'])} keys")

    # --- Checkpoint directory for this run ---
    run_dir = f"{master_output_dir}/epoch_{ckpt_epoch}"
    Path(run_dir).mkdir(parents=True, exist_ok=True)
    save_path = f"{run_dir}/best_model.pt"
    integration_snapshot_path = f"{run_dir}/integration_snapshots"
    os.makedirs(integration_snapshot_path, exist_ok=True)

    # --- Create optimizer + scheduler ---
    optimizer = create_stream_optimizer(
        model,
        optimizer_type='adamw',
        stream_lrs=SUN_OPTIMIZER_CONFIG['stream_lrs'],
        stream_weight_decays=SUN_OPTIMIZER_CONFIG['stream_weight_decays'],
        shared_lr=SUN_OPTIMIZER_CONFIG['shared_lr'],
        integration_weight_decay=SUN_OPTIMIZER_CONFIG['integration_weight_decay'],
        stem_lr_multiplier=SUN_OPTIMIZER_CONFIG['stem_lr_multiplier'],
    )

    scheduler = setup_scheduler(
        optimizer,
        scheduler_type=SUN_SCHEDULER_CONFIG['scheduler_type'],
        epochs=SUN_SCHEDULER_CONFIG['t_max'],
        train_loader_len=len(sun_train_loader),
        t_max=SUN_SCHEDULER_CONFIG['t_max'],
        eta_min=(
            ([SUN_SCHEDULER_CONFIG['s1_eta'] * SUN_OPTIMIZER_CONFIG['stem_lr_multiplier'],
              SUN_SCHEDULER_CONFIG['s2_eta'] * SUN_OPTIMIZER_CONFIG['stem_lr_multiplier']]
             if SUN_OPTIMIZER_CONFIG['stem_lr_multiplier'] != 1.0 else []) +
            [SUN_SCHEDULER_CONFIG['s1_eta'], SUN_SCHEDULER_CONFIG['s2_eta'],
             SUN_SCHEDULER_CONFIG['eta_min'], SUN_SCHEDULER_CONFIG['eta_min']]
        ),
        warmup_epochs=SUN_SCHEDULER_CONFIG['warmup_epochs'],
        warmup_start_factor=SUN_SCHEDULER_CONFIG['warmup_start_factor'],
    )

    # --- Compile ---
    model.compile(
        optimizer=optimizer,
        scheduler=scheduler,
        loss='cross_entropy',
        label_smoothing=SUN_TRAIN_CONFIG['label_smoothing'],
        gpu_augmentation=False,
        **SUN_AUGMENTATION_CONFIG.to_dict(),
    )

    # --- Optional backbone freeze warmup ---
    warmup_history = None
    freeze_epochs = TRANSFER_CONFIG['freeze_backbone_epochs']
    if freeze_epochs > 0:
        print(f"\nBackbone freeze warmup ({freeze_epochs} epochs)...")
        for name, param in model.named_parameters():
            if not name.startswith('fc.'):
                param.requires_grad = False

        fc_optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=TRANSFER_CONFIG['freeze_backbone_lr'],
        )
        model.compile(
            optimizer=fc_optimizer, scheduler=None,
            loss='cross_entropy',
            label_smoothing=SUN_TRAIN_CONFIG['label_smoothing'],
            gpu_augmentation=False,
            **SUN_AUGMENTATION_CONFIG.to_dict(),
        )
        warmup_history = model.fit(
            train_loader=sun_train_loader, val_loader=sun_val_loader,
            epochs=freeze_epochs, verbose=True,
            grad_clip_norm=SUN_TRAIN_CONFIG['grad_clip_norm'],
            stream_monitoring=False, modality_dropout=False,
            gradient_monitoring=False, track_integration_weights=False,
        )

        for param in model.parameters():
            param.requires_grad = True

        # Re-compile with full optimizer
        optimizer = create_stream_optimizer(
            model, optimizer_type='adamw',
            stream_lrs=SUN_OPTIMIZER_CONFIG['stream_lrs'],
            stream_weight_decays=SUN_OPTIMIZER_CONFIG['stream_weight_decays'],
            shared_lr=SUN_OPTIMIZER_CONFIG['shared_lr'],
            integration_weight_decay=SUN_OPTIMIZER_CONFIG['integration_weight_decay'],
            stem_lr_multiplier=SUN_OPTIMIZER_CONFIG['stem_lr_multiplier'],
        )
        scheduler = setup_scheduler(
            optimizer,
            scheduler_type=SUN_SCHEDULER_CONFIG['scheduler_type'],
            epochs=SUN_SCHEDULER_CONFIG['t_max'],
            train_loader_len=len(sun_train_loader),
            t_max=SUN_SCHEDULER_CONFIG['t_max'],
            eta_min=(
                ([SUN_SCHEDULER_CONFIG['s1_eta'] * SUN_OPTIMIZER_CONFIG['stem_lr_multiplier'],
                  SUN_SCHEDULER_CONFIG['s2_eta'] * SUN_OPTIMIZER_CONFIG['stem_lr_multiplier']]
                 if SUN_OPTIMIZER_CONFIG['stem_lr_multiplier'] != 1.0 else []) +
                [SUN_SCHEDULER_CONFIG['s1_eta'], SUN_SCHEDULER_CONFIG['s2_eta'],
                 SUN_SCHEDULER_CONFIG['eta_min'], SUN_SCHEDULER_CONFIG['eta_min']]
            ),
            warmup_epochs=SUN_SCHEDULER_CONFIG['warmup_epochs'],
            warmup_start_factor=SUN_SCHEDULER_CONFIG['warmup_start_factor'],
        )
        model.compile(
            optimizer=optimizer, scheduler=scheduler,
            loss='cross_entropy',
            label_smoothing=SUN_TRAIN_CONFIG['label_smoothing'],
            gpu_augmentation=False,
            **SUN_AUGMENTATION_CONFIG.to_dict(),
        )

    # --- Train ---
    print(f"\n{'='*60}")
    print(f"SUN FINE-TUNING (from ScanNet epoch {ckpt_epoch})")
    print(f"{'='*60}")

    sun_history = model.fit(
        train_loader=sun_train_loader,
        val_loader=sun_val_loader,
        epochs=SUN_TRAIN_CONFIG['epochs'],
        verbose=True,
        save_path=save_path,
        early_stopping=SUN_TRAIN_CONFIG['early_stopping'],
        restore_best_weights=SUN_TRAIN_CONFIG['restore_best_weights'],
        grad_clip_norm=SUN_TRAIN_CONFIG['grad_clip_norm'],
        stream_monitoring=SUN_TRAIN_CONFIG['stream_monitoring'],
        monitor=SUN_TRAIN_CONFIG['monitor'],
        modality_dropout=SUN_TRAIN_CONFIG['modality_dropout'],
        modality_dropout_start=SUN_TRAIN_CONFIG['modality_dropout_start'],
        modality_dropout_ramp=SUN_TRAIN_CONFIG['modality_dropout_ramp'],
        modality_dropout_rate=SUN_TRAIN_CONFIG['modality_dropout_rate'],
        gradient_monitoring=SUN_TRAIN_CONFIG['gradient_monitoring'],
        gradient_log_freq=SUN_TRAIN_CONFIG['gradient_log_freq'],
        track_integration_weights=SUN_TRAIN_CONFIG['track_integration_weights'],
        integration_snapshot_path=integration_snapshot_path,
        integration_snapshot_freq=SUN_TRAIN_CONFIG['integration_snapshot_freq'],
    )

    # Merge warmup + full history
    if warmup_history is not None:
        history = {}
        all_keys = set(warmup_history.keys()) | set(sun_history.keys())
        for key in all_keys:
            v1, v2 = warmup_history.get(key), sun_history.get(key)
            if isinstance(v1, list) and isinstance(v2, list):
                history[key] = v1 + v2
            elif v2 is not None:
                history[key] = v2
            else:
                history[key] = v1
    else:
        history = sun_history

    # --- Evaluate on validation set ---
    print(f"\n{'='*60}")
    print(f"VALIDATION SET EVALUATION (ScanNet epoch {ckpt_epoch})")
    print(f"{'='*60}")

    results = model.evaluate(data_loader=sun_val_loader, stream_monitoring=True)

    print(f"  Overall Accuracy: {results['accuracy']*100:.2f}%")
    print(f"  Mean Class Accuracy: {results['mean_class_accuracy']*100:.2f}%")
    print(f"  Loss: {results['loss']:.4f}")

    for i in range(len(SUN_MODEL_CONFIG['stream_input_channels'])):
        other = (i + 1) % 2
        solo_acc = results[f'stream_{other}_blanked_acc']
        print(f"  {STREAM_LABELS[i]} Solo Accuracy: {solo_acc*100:.2f}%")

    # --- Save results ---
    # Save final model
    final_path = f"{run_dir}/final_model.pt"
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': model.optimizer.state_dict(),
        'scheduler_state_dict': model.scheduler.state_dict() if model.scheduler else None,
        'config': SUN_MODEL_CONFIG,
        'history': history,
        'val_accuracy': results['accuracy'],
        'val_mca': results['mean_class_accuracy'],
        'source_checkpoint_epoch': ckpt_epoch,
    }, final_path)

    # Save history JSON
    history_path = f"{run_dir}/training_history.json"
    with open(history_path, 'w') as f:
        json_history = {
            'source_checkpoint_epoch': ckpt_epoch,
            'train_loss': [float(x) for x in history['train_loss']],
            'train_accuracy': [float(x) for x in history['train_accuracy']],
            'train_mca': [float(x) for x in history.get('train_mca', [])],
            'val_loss': [float(x) for x in history.get('val_loss', [])],
            'val_accuracy': [float(x) for x in history.get('val_accuracy', [])],
            'val_mca': [float(x) for x in history.get('val_mca', [])],
            'learning_rates': [float(x) for x in history['learning_rates']],
            'val_results': {
                'loss': float(results['loss']),
                'accuracy': float(results['accuracy']),
                'mean_class_accuracy': float(results.get('mean_class_accuracy', 0)),
            },
            'model_config': SUN_MODEL_CONFIG,
            'optimizer_config': SUN_OPTIMIZER_CONFIG,
            'scheduler_config': SUN_SCHEDULER_CONFIG,
            'training_config': {k: str(v) if not isinstance(v, (int, float, bool, type(None))) else v
                                for k, v in SUN_TRAIN_CONFIG.items()},
            'augmentation_config': SUN_AUGMENTATION_CONFIG.to_dict(),
        }
        json.dump(json_history, f, indent=2)

    # Collect summary
    all_results[ckpt_epoch] = {
        'val_accuracy': results['accuracy'],
        'val_mca': results['mean_class_accuracy'],
        'val_loss': results['loss'],
        'final_train_loss': history['train_loss'][-1],
        'final_train_acc': history['train_accuracy'][-1],
        'run_dir': run_dir,
    }

    print(f"\nResults saved to: {run_dir}")

    # Free GPU memory before next run
    del model, optimizer, scheduler
    torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("ALL CHECKPOINT RUNS COMPLETE!")
print("=" * 60)

## 10. Comparison Summary

In [ ]:
print("=" * 70)
print("TRANSFER LEARNING ABLATION: ScanNet Epoch vs SUN Val Performance")
print("=" * 70)

print(f"\n{'Epoch':>8} | {'Val Acc':>10} | {'Val MCA':>10} | {'Val Loss':>10} | {'Train Loss':>10}")
print("-" * 70)

best_mca = 0
best_epoch = None
for epoch in sorted(all_results.keys()):
    r = all_results[epoch]
    print(f"{epoch:>8} | {r['val_accuracy']*100:>9.2f}% | {r['val_mca']*100:>9.2f}% | "
          f"{r['val_loss']:>10.4f} | {r['final_train_loss']:>10.4f}")
    if r['val_mca'] > best_mca:
        best_mca = r['val_mca']
        best_epoch = epoch

print("-" * 70)
print(f"\nBest Val MCA: {best_mca*100:.2f}% from ScanNet epoch {best_epoch}")

# Save summary
summary_path = f"{master_output_dir}/ablation_summary.json"
summary = {
    'checkpoint_epochs': sorted(all_results.keys()),
    'results': {str(k): {kk: float(vv) if isinstance(vv, float) else vv
                          for kk, vv in v.items()}
                for k, v in all_results.items()},
    'best_epoch': best_epoch,
    'best_val_mca': float(best_mca),
    'sun_config': {
        'epochs': SUN_TRAIN_CONFIG['epochs'],
        'batch_size': SUN_DATASET_CONFIG['batch_size'],
    },
}
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\nSummary saved: {summary_path}")
print(f"All results: {master_output_dir}")

## 11. Comparison Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_sorted = sorted(all_results.keys())
val_accs = [all_results[e]['val_accuracy'] * 100 for e in epochs_sorted]
val_mcas = [all_results[e]['val_mca'] * 100 for e in epochs_sorted]
val_losses = [all_results[e]['val_loss'] for e in epochs_sorted]

# Val Accuracy
axes[0].bar(range(len(epochs_sorted)), val_accs, color='steelblue', alpha=0.8)
axes[0].set_xticks(range(len(epochs_sorted)))
axes[0].set_xticklabels([f'Epoch {e}' for e in epochs_sorted])
axes[0].set_ylabel('Val Accuracy (%)')
axes[0].set_title('Val Accuracy by Pretrain Epoch')
for i, v in enumerate(val_accs):
    axes[0].text(i, v + 0.2, f'{v:.1f}%', ha='center', fontsize=10)

# Val MCA
axes[1].bar(range(len(epochs_sorted)), val_mcas, color='darkorange', alpha=0.8)
axes[1].set_xticks(range(len(epochs_sorted)))
axes[1].set_xticklabels([f'Epoch {e}' for e in epochs_sorted])
axes[1].set_ylabel('Val MCA (%)')
axes[1].set_title('Val Mean Class Accuracy by Pretrain Epoch')
for i, v in enumerate(val_mcas):
    axes[1].text(i, v + 0.2, f'{v:.1f}%', ha='center', fontsize=10)

# Val Loss
axes[2].bar(range(len(epochs_sorted)), val_losses, color='forestgreen', alpha=0.8)
axes[2].set_xticks(range(len(epochs_sorted)))
axes[2].set_xticklabels([f'Epoch {e}' for e in epochs_sorted])
axes[2].set_ylabel('Val Loss')
axes[2].set_title('Val Loss by Pretrain Epoch')
for i, v in enumerate(val_losses):
    axes[2].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

plt.suptitle('ScanNet Pretrain Epoch vs SUN Transfer Performance (Validation)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{master_output_dir}/epoch_ablation_comparison.pdf', bbox_inches='tight', dpi=150)
plt.show()
print(f"Plot saved: {master_output_dir}/epoch_ablation_comparison.pdf")

## 12. Summary

**Saved per checkpoint:**
- `best_model.pt` - Best model checkpoint (by train loss, since no val set)
- `final_model.pt` - Final model with full state dict, optimizer, scheduler, history
- `training_history.json` - Full training history + test results

**Saved overall:**
- `ablation_summary.json` - Comparison of all checkpoint runs
- `epoch_ablation_comparison.pdf` - Bar plots comparing test performance